# MiniSense — A Survey Analysis Agent

A small, runnable multi-agent system that answers business questions about
survey feedback using an **Orchestrator + Sub-Agents** architecture
(LangGraph) and a **document-grounded RAG pipeline** (LangChain + FAISS)
over a product FAQ.

**Contents**
1. Setup
2. Synthetic survey data (Appendix A)
3. Product FAQ document (Appendix B, expanded)
4. Part 2 — RAG pipeline: chunk → embed → store → retrieve
5. Part 1 — Sub-agents: DataAgent, RAGAgent, ComparisonAgent, SummaryAgent
6. Part 1 — Orchestrator (LangGraph)
7. Evaluation checkpoint — 3 sample questions
8. Part 3 — Fine-tuning design write-up


## 1. Setup

Install the frameworks used in this notebook. Everything here runs
**offline** — no API key and no model download required — so it works in
any grading environment. Sub-section 5.4 shows how to plug in a real LLM
(OpenAI or Anthropic) if a key is set, and Section 4 documents how to swap
in real sentence embeddings later.


In [1]:
# !pip install -q langchain langgraph langchain-community faiss-cpu scikit-learn numpy pandas
# Uncomment the line above the first time you run this in a fresh environment.


In [2]:
import json
import random
import datetime
import statistics
import os
import re
from dataclasses import dataclass, asdict
from typing import List, Dict, Optional, TypedDict

random.seed(42)  # reproducible synthetic data


## 2. Synthetic Survey Data (Appendix A)

The assignment leaves data generation to the candidate: *"how you construct
test data reflects your thinking."* The approach here:

- **Rating distribution** is weighted (more 4s/5s than 1s), which mirrors
  typical real-world CSAT survey skew rather than a uniform random spread.
- **`free_text` is theme-conditioned on the rating** — positive ratings pull
  positive phrases, low ratings pull negative phrases — so metrics computed
  from `rating` and themes extracted from `free_text` actually agree with
  each other, which matters for testing the DataAgent honestly.
- ~40% of responses mention a **second theme** for realism (e.g. "food was
  great, but the wait was too long"), which also creates real disagreement
  for the theme-extraction logic to handle.
- Dates span two calendar months so the ComparisonAgent has two clean
  periods to compare.

Scale: the brief asks for 50,000–100,000 rows. The notebook defaults to
20,000 for fast iteration; set `N_RESPONSES` below to 100000 for the full
submission run — generation and metric computation are O(n) and stay fast
at that size (FAISS/RAG only indexes the small FAQ doc, not the survey
data, so this doesn't affect retrieval cost).


In [3]:
N_RESPONSES = 20000  # bump to 100000 for the full-scale submission run

BUSINESSES = [
    ("b01", "QuickFit Gym"),
    ("b02", "GreenLeaf Bistro"),
    ("b03", "UrbanBrew Cafe"),
]
SURVEYS = ["Membership Value", "Dine-in Experience", "Delivery Feedback"]
CHANNELS = ["mobile", "web", "kiosk", "email"]

THEMES = {
    "food": ["the food", "the menu", "the coffee", "the avocado toast", "the garden bowl"],
    "wait_time": ["the wait time", "how long we waited", "the queue", "service speed"],
    "staff": ["the staff", "the barista", "the manager", "the team"],
    "price": ["the pricing", "the cost", "the membership fee", "value for money"],
    "cleanliness": ["the cleanliness", "the tables", "the restrooms", "the gym floor"],
}

POSITIVE_PHRASES = ["was great", "was excellent", "exceeded expectations", "was on point", "impressed me"]
NEGATIVE_PHRASES = ["was too long", "was disappointing", "needs improvement", "was frustrating", "let me down"]
NEUTRAL_PHRASES = ["was okay", "was average", "was fine, nothing special"]


def _free_text(rating: int) -> str:
    theme = random.choice(list(THEMES.keys()))
    subject = random.choice(THEMES[theme])
    if rating >= 4:
        phrase = random.choice(POSITIVE_PHRASES)
    elif rating == 3:
        phrase = random.choice(NEUTRAL_PHRASES)
    else:
        phrase = random.choice(NEGATIVE_PHRASES)
    if random.random() < 0.4:  # occasional second clause, for realism
        theme2 = random.choice(list(THEMES.keys()))
        subject2 = random.choice(THEMES[theme2])
        phrase2 = random.choice(POSITIVE_PHRASES if random.random() < 0.5 else NEGATIVE_PHRASES)
        return f"{subject.capitalize()} {phrase}, but {subject2} {phrase2}."
    return f"{subject.capitalize()} {phrase}."


def generate_survey_data(n: int, start: str = "2026-04-01", end: str = "2026-05-31") -> Dict:
    start_d = datetime.date.fromisoformat(start)
    end_d = datetime.date.fromisoformat(end)
    span = (end_d - start_d).days
    responses = []
    for i in range(n):
        biz_id, biz_name = random.choice(BUSINESSES)
        rating = random.choices([1, 2, 3, 4, 5], weights=[8, 10, 17, 30, 35])[0]
        day = start_d + datetime.timedelta(days=random.randint(0, span))
        responses.append({
            "response_id": f"r{i:06d}",
            "date": day.isoformat(),
            "business_id": biz_id,
            "business_name": biz_name,
            "survey_id": f"s{random.randint(1, 3):02d}",
            "survey_name": random.choice(SURVEYS),
            "rating": rating,
            "response_channel": random.choice(CHANNELS),
            "free_text": _free_text(rating),
        })
    return {"responses": responses}


survey_data = generate_survey_data(N_RESPONSES)
with open("survey_data.json", "w") as f:
    json.dump(survey_data, f)

print(f"Generated {len(survey_data['responses'])} responses")
print(json.dumps(survey_data["responses"][0], indent=2))


Generated 20000 responses
{
  "response_id": "r000000",
  "date": "2026-05-18",
  "business_id": "b03",
  "business_name": "UrbanBrew Cafe",
  "survey_id": "s02",
  "survey_name": "Membership Value",
  "rating": 2,
  "response_channel": "web",
  "free_text": "The wait time let me down, but the pricing was excellent."
}


## 3. Product FAQ Document (Appendix B, expanded)

Expanded from the ~90-word sample to ~500 words by adding FAQ entries that
the sub-agents actually need context for later: feedback channels, food
safety escalation, and pricing policy — so retrieval has real material to
work with beyond the four seed questions.


In [4]:
FAQ_TEXT = """GreenLeaf Bistro -- Customer Experience FAQ

Q: What are your most popular menu items?
A: Our top sellers are the Avocado Toast, Garden Bowl, and Cold Brew Coffee. Seasonal items rotate every quarter based on customer feedback and local produce availability.

Q: What is your average wait time?
A: We target under 10 minutes for counter orders during off-peak hours. Peak hours (12-1 PM, 6-8 PM) may see 15-20 minute waits. During peak season (May-August), wait times can increase by an additional 5 minutes on average.

Q: How do you handle complaints?
A: All complaints are escalated to the shift manager within 15 minutes. Refunds or replacements are offered for quality issues. Repeated complaints from the same customer within 30 days trigger a personal follow-up call from the regional manager.

Q: What is your CSAT target?
A: We aim for a CSAT of 4.5+. Scores below 4.0 trigger a root-cause review with the operations team. Reviews below 3.5 for two consecutive weeks trigger a full site audit.

Q: What are your operating hours?
A: Locations are open 7 AM to 9 PM on weekdays and 8 AM to 10 PM on weekends. Holiday hours are posted a week in advance on our app and website.

Q: Do you offer delivery?
A: Yes, delivery is available through our app and major third-party platforms within a 5-mile radius. Delivery orders are tracked separately from in-store CSAT because wait-time expectations differ.

Q: How do you train staff on customer service?
A: New hires complete a 2-week onboarding covering food safety, POS systems, and de-escalation techniques for handling frustrated customers. Refresher training happens quarterly.

Q: What membership or loyalty programs do you offer?
A: Our loyalty app rewards customers with a free item after every 10 purchases. Membership customers at partner gyms (like QuickFit) receive a 10% discount on in-store orders.

Q: How is staffing adjusted for busy periods?
A: Staffing schedules are built two weeks in advance using historical order volume. During observed spikes in wait-time complaints, shift leads can call in on-call staff with 2 hours notice.

Q: How do you use customer survey data?
A: Survey responses are reviewed weekly by the operations team. Recurring themes -- such as wait time or food quality -- are prioritized in the following sprint's operational changes, and CSAT trends are shared company-wide in the monthly business review.

Q: What channels can customers use to leave feedback?
A: Customers can respond via the mobile app, our website, an in-store kiosk, or an email survey sent 24 hours after a purchase. Mobile responses make up the majority of our feedback volume, followed by kiosk and web.

Q: How do you handle food quality issues specifically?
A: Any complaint mentioning spoiled, cold, or incorrect food is flagged automatically for same-day manager review, separate from the standard 15-minute escalation window, since food safety issues carry additional reporting requirements.

Q: What is your policy on price changes?
A: Menu prices are reviewed twice a year. Any increase above 5% is communicated in-app two weeks in advance, and loyalty members are grandfathered into prior pricing for their next three visits.
"""

with open("faq.txt", "w") as f:
    f.write(FAQ_TEXT)

print(f"FAQ document: {len(FAQ_TEXT.split())} words")


FAQ document: 520 words


## 4. Part 2 — RAG Pipeline

**Chunking strategy — sentence/paragraph-aware, not fixed-size.**
The FAQ is a Q&A document, so the natural semantic unit is one Q&A pair,
not an arbitrary N-character window. `RecursiveCharacterTextSplitter`
tries paragraph breaks first, then sentence breaks, only falling back to
raw character windows if a chunk is still too long. This keeps each
retrieved chunk self-contained and directly answerable, which matters more
here than hitting an exact token budget — a fixed-size splitter would risk
cutting a question off from its answer mid-chunk.

**Embedding model — TF-IDF, not sentence-transformers/OpenAI.**
A real sentence-embedding model needs either a model download or an API
key, which isn't guaranteed to be available in every grading environment.
TF-IDF needs neither, is instant on a document this small, and captures
keyword-level overlap well enough for a ~13-chunk FAQ. This is a scope
trade-off, not a claim that TF-IDF beats real embeddings — Section 7 shows
exactly where this choice falls short. Swapping in real embeddings is a
one-line change (commented below).

**Vector store — FAISS**, as suggested by the brief.


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.embeddings import Embeddings
from sklearn.feature_extraction.text import TfidfVectorizer

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=40,
    separators=["\n\n", "\n", ". ", " "],
)
faq_chunks = [c for c in splitter.split_text(FAQ_TEXT) if c.strip()]
print(f"FAQ split into {len(faq_chunks)} chunks")
print("\nExample chunk:\n", faq_chunks[1])


/tmp/ipykernel_615/1712016731.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


FAQ split into 13 chunks

Example chunk:
 Q: What is your average wait time?
A: We target under 10 minutes for counter orders during off-peak hours. Peak hours (12-1 PM, 6-8 PM) may see 15-20 minute waits. During peak season (May-August), wait times can increase by an additional 5 minutes on average.


In [6]:
class TfidfEmbeddings(Embeddings):
    """Offline embedding function built on TF-IDF (see markdown above for why)."""

    def __init__(self, corpus: List[str]):
        self.vectorizer = TfidfVectorizer(stop_words="english")
        self.vectorizer.fit(corpus)

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        return self.vectorizer.transform(texts).toarray().tolist()

    def embed_query(self, text: str) -> List[float]:
        return self.vectorizer.transform([text]).toarray()[0].tolist()


# To use real embeddings instead, swap the two lines below for e.g.:
#   from langchain_community.embeddings import HuggingFaceEmbeddings
#   embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
# or
#   from langchain_openai import OpenAIEmbeddings
#   embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
embeddings = TfidfEmbeddings(faq_chunks)

vectorstore = FAISS.from_texts(faq_chunks, embeddings)
print("FAISS index built with", vectorstore.index.ntotal, "vectors")

# quick retrieval smoke test
for chunk in vectorstore.similarity_search("what is the CSAT target", k=2):
    print("-", chunk.page_content.replace(chr(10), " ")[:90])


FAISS index built with 13 vectors
- Q: What is your CSAT target? A: We aim for a CSAT of 4.5+. Scores below 4.0 trigger a root
- Q: What is your average wait time? A: We target under 10 minutes for counter orders during


## 5. Part 1 — Sub-Agents

Each sub-agent takes a **structured `TaskSpec`** (not raw text) from the
orchestrator and returns a **structured dataclass** (not free text) —
per the assignment's requirements.

### 5.1 Shared types


In [7]:
@dataclass
class TaskSpec:
    """Structured instruction the Orchestrator sends to a sub-agent."""
    task_type: str                     # "metrics" | "retrieve" | "compare" | "summarize"
    query: str = ""
    start_date: Optional[str] = None
    end_date: Optional[str] = None
    business_name: Optional[str] = None
    top_k: int = 3


@dataclass
class DataAgentResult:
    response_count: int
    avg_rating: float
    csat: float                        # % of responses rated 4 or 5
    top_themes: List[str]


@dataclass
class RAGAgentResult:
    chunks: List[str]
    scores: List[float]


@dataclass
class ComparisonAgentResult:
    current: DataAgentResult
    previous: DataAgentResult
    csat_delta: float
    rating_delta: float
    notable_change: str


### 5.2 DataAgent — parses survey JSON, computes exact metrics

Includes the assignment's required **explicit tool-calling example**:
`compute_csat()` is a standalone function the agent *calls*, rather than
inlining the calculation — the same pattern any of these agents would use
to call a real external tool (a DB query, an API) in production.


In [8]:
# --- Tool: DataAgent calls this rather than computing CSAT inline ---
def compute_csat(ratings: List[int]) -> float:
    """CSAT = % of responses with rating >= 4."""
    if not ratings:
        return 0.0
    satisfied = sum(1 for r in ratings if r >= 4)
    return round(100 * satisfied / len(ratings), 1)


def extract_top_themes(free_texts: List[str], top_n: int = 3) -> List[str]:
    """Small keyword counter over the fixed THEMES vocabulary."""
    counts = {theme: 0 for theme in THEMES}
    for text in free_texts:
        low = text.lower()
        for theme, subjects in THEMES.items():
            if any(s.split()[-1] in low for s in subjects):
                counts[theme] += 1
    ranked = sorted(counts.items(), key=lambda x: x[1], reverse=True)
    return [t for t, c in ranked[:top_n] if c > 0]


class DataAgent:
    """Parses the survey JSON and computes exact metrics for a task spec."""

    def __init__(self, responses: List[Dict]):
        self.responses = responses

    def _filter(self, spec: TaskSpec) -> List[Dict]:
        rows = self.responses
        if spec.start_date:
            rows = [r for r in rows if r["date"] >= spec.start_date]
        if spec.end_date:
            rows = [r for r in rows if r["date"] <= spec.end_date]
        if spec.business_name:
            rows = [r for r in rows if r["business_name"] == spec.business_name]
        return rows

    def run(self, spec: TaskSpec) -> DataAgentResult:
        rows = self._filter(spec)
        ratings = [r["rating"] for r in rows]
        texts = [r["free_text"] for r in rows]
        return DataAgentResult(
            response_count=len(rows),
            avg_rating=round(statistics.mean(ratings), 2) if ratings else 0.0,
            csat=compute_csat(ratings),          # <-- tool call
            top_themes=extract_top_themes(texts),
        )


# smoke test
_test_agent = DataAgent(survey_data["responses"])
print(_test_agent.run(TaskSpec(task_type="metrics", start_date="2026-05-01", end_date="2026-05-31")))


DataAgentResult(response_count=10227, avg_rating=3.72, csat=63.8, top_themes=['price', 'wait_time', 'food'])


### 5.3 RAGAgent — retrieves relevant FAQ chunks for business context


In [9]:
class RAGAgent:
    """Retrieves top-k relevant FAQ chunks for a query."""

    def __init__(self, vectorstore: FAISS):
        self.vectorstore = vectorstore

    def run(self, spec: TaskSpec) -> RAGAgentResult:
        hits = self.vectorstore.similarity_search_with_score(spec.query, k=spec.top_k)
        chunks = [h[0].page_content for h in hits]
        scores = [round(float(h[1]), 3) for h in hits]
        return RAGAgentResult(chunks=chunks, scores=scores)


_test_rag = RAGAgent(vectorstore)
print(_test_rag.run(TaskSpec(task_type="retrieve", query="what is the CSAT target", top_k=2)))


RAGAgentResult(chunks=['Q: What is your CSAT target?\nA: We aim for a CSAT of 4.5+. Scores below 4.0 trigger a root-cause review with the operations team. Reviews below 3.5 for two consecutive weeks trigger a full site audit.', 'Q: What is your average wait time?\nA: We target under 10 minutes for counter orders during off-peak hours. Peak hours (12-1 PM, 6-8 PM) may see 15-20 minute waits. During peak season (May-August), wait times can increase by an additional 5 minutes on average.'], scores=[1.188, 1.782])


### 5.4 ComparisonAgent — compares two time periods


In [10]:
class ComparisonAgent:
    """Compares two time periods using the DataAgent under the hood."""

    def __init__(self, data_agent: DataAgent):
        self.data_agent = data_agent

    def run(self, current_spec: TaskSpec, previous_spec: TaskSpec) -> ComparisonAgentResult:
        current = self.data_agent.run(current_spec)
        previous = self.data_agent.run(previous_spec)
        csat_delta = round(current.csat - previous.csat, 1)
        rating_delta = round(current.avg_rating - previous.avg_rating, 2)
        direction = "improved" if csat_delta > 0 else ("declined" if csat_delta < 0 else "held steady")
        notable_change = f"CSAT {direction} by {abs(csat_delta)} points month-over-month."
        return ComparisonAgentResult(current, previous, csat_delta, rating_delta, notable_change)


### 5.5 SummaryAgent — drafts the final narrative paragraph

Uses a real LLM (Anthropic or OpenAI — whichever API key is set in the
environment) when available, and falls back to a deterministic template
otherwise. This keeps the notebook fully runnable with zero setup while
still supporting the "coherent narrative, not raw numbers" requirement
either way.


In [11]:
USE_LLM = bool(os.environ.get("OPENAI_API_KEY") or os.environ.get("ANTHROPIC_API_KEY"))

if USE_LLM:
    if os.environ.get("ANTHROPIC_API_KEY"):
        from langchain_anthropic import ChatAnthropic
        llm = ChatAnthropic(model="claude-sonnet-4-6", temperature=0.2)
    else:
        from langchain_openai import ChatOpenAI
        llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)


class SummaryAgent:
    """Synthesizes DataAgent / RAGAgent / ComparisonAgent output into one paragraph."""

    def run(self, question: str, data: Optional[DataAgentResult] = None,
            rag: Optional[RAGAgentResult] = None,
            comparison: Optional[ComparisonAgentResult] = None) -> str:
        if USE_LLM:
            return self._run_llm(question, data, rag, comparison)
        return self._run_template(question, data, rag, comparison)

    def _run_llm(self, question, data, rag, comparison) -> str:
        context_parts = [f"Business question: {question}"]
        if data:
            context_parts.append(f"Survey metrics: {asdict(data)}")
        if comparison:
            context_parts.append(
                f"Comparison: current={asdict(comparison.current)}, "
                f"previous={asdict(comparison.previous)}, "
                f"csat_delta={comparison.csat_delta}, notable_change={comparison.notable_change}"
            )
        if rag:
            context_parts.append("Relevant FAQ context:\n" + "\n---\n".join(rag.chunks))
        prompt = (
            "You are a business analyst. Using ONLY the data below, write one "
            "short, coherent narrative paragraph answering the question. "
            "Reference concrete numbers where available.\n\n" + "\n\n".join(context_parts)
        )
        return llm.invoke(prompt).content

    def _run_template(self, question, data, rag, comparison) -> str:
        parts = []
        if comparison:
            c, p = comparison.current, comparison.previous
            parts.append(
                f"In the current period, CSAT was {c.csat}% (avg rating {c.avg_rating}) "
                f"across {c.response_count} responses, versus {p.csat}% "
                f"(avg rating {p.avg_rating}) in the previous period -- {comparison.notable_change}"
            )
            if c.top_themes:
                parts.append(f"The most common themes this period were {', '.join(c.top_themes)}.")
        elif data:
            parts.append(
                f"Across {data.response_count} responses, CSAT is {data.csat}% "
                f"with an average rating of {data.avg_rating}/5."
            )
            if data.top_themes:
                parts.append(f"The most frequently mentioned themes were {', '.join(data.top_themes)}.")
        if rag and rag.chunks:
            faq_note = rag.chunks[0].replace(chr(10), " ")
            parts.append(f'For context, company policy notes: "{faq_note}"')
        if not parts:
            parts.append("No data was available to answer this question.")
        return " ".join(parts)

print("SummaryAgent ready. USE_LLM =", USE_LLM,
      "(set OPENAI_API_KEY or ANTHROPIC_API_KEY to use a real LLM instead of the template)")


SummaryAgent ready. USE_LLM = False (set OPENAI_API_KEY or ANTHROPIC_API_KEY to use a real LLM instead of the template)


## 6. Part 1 — Orchestrator Agent (LangGraph)

The Orchestrator is a `StateGraph`:

```
planner → [comparison_agent | data_agent] → [rag_agent] → summary_agent → END
```

- **`planner`** breaks the question into `TaskSpec`s and decides, with a
  small keyword rule set, whether the question needs `ComparisonAgent`
  and/or `RAGAgent`. This is intentionally rule-based rather than an LLM
  call — the routing decision itself is small and deterministic enough
  that an LLM would add latency and cost without adding accuracy; an
  LLM-based planner is the natural upgrade once the rule set outgrows a
  handful of keywords.
- **`data_agent` / `comparison_agent`** always run (every question needs
  some metric), with comparison replacing plain metrics when the question
  is about change over time.
- **`rag_agent`** runs conditionally, only when the question needs FAQ
  context (e.g. mentions "policy", "target", "complaint").
- **`summary_agent`** always runs last and synthesizes whatever upstream
  state is present into the final narrative.


In [12]:
from langgraph.graph import StateGraph, END

data_agent = DataAgent(survey_data["responses"])
rag_agent = RAGAgent(vectorstore)
comparison_agent = ComparisonAgent(data_agent)
summary_agent = SummaryAgent()


class GraphState(TypedDict, total=False):
    question: str
    needs_comparison: bool
    needs_rag: bool
    current_spec: TaskSpec
    previous_spec: TaskSpec
    rag_spec: TaskSpec
    data_result: DataAgentResult
    comparison_result: ComparisonAgentResult
    rag_result: RAGAgentResult
    final_answer: str


# Current / previous month, matching the two-month span of the generated data
DATE_RANGES = {
    "current": ("2026-05-01", "2026-05-31"),
    "previous": ("2026-04-01", "2026-04-30"),
}


def planner_node(state: GraphState) -> GraphState:
    q = state["question"].lower()
    needs_comparison = any(w in q for w in ["compare", "last month", "vs", "versus", "change"])
    needs_rag = any(w in q for w in ["policy", "target", "faq", "handle", "wait",
                                      "csat target", "complaint", "why", "should"])

    cur_start, cur_end = DATE_RANGES["current"]
    prev_start, prev_end = DATE_RANGES["previous"]
    state["needs_comparison"] = needs_comparison
    state["needs_rag"] = needs_rag
    state["current_spec"] = TaskSpec(task_type="metrics", start_date=cur_start, end_date=cur_end)
    state["previous_spec"] = TaskSpec(task_type="metrics", start_date=prev_start, end_date=prev_end)
    state["rag_spec"] = TaskSpec(task_type="retrieve", query=state["question"], top_k=3)
    return state


def data_node(state: GraphState) -> GraphState:
    state["data_result"] = data_agent.run(state["current_spec"])
    return state


def comparison_node(state: GraphState) -> GraphState:
    state["comparison_result"] = comparison_agent.run(state["current_spec"], state["previous_spec"])
    return state


def rag_node(state: GraphState) -> GraphState:
    state["rag_result"] = rag_agent.run(state["rag_spec"])
    return state


def summary_node(state: GraphState) -> GraphState:
    state["final_answer"] = summary_agent.run(
        question=state["question"],
        data=state.get("data_result"),
        rag=state.get("rag_result"),
        comparison=state.get("comparison_result"),
    )
    return state


def route_after_planner(state: GraphState) -> str:
    return "comparison_agent" if state["needs_comparison"] else "data_agent"


def route_after_metrics(state: GraphState) -> str:
    return "rag_agent" if state["needs_rag"] else "summary_agent"


graph = StateGraph(GraphState)
graph.add_node("planner", planner_node)
graph.add_node("data_agent", data_node)
graph.add_node("comparison_agent", comparison_node)
graph.add_node("rag_agent", rag_node)
graph.add_node("summary_agent", summary_node)

graph.set_entry_point("planner")
graph.add_conditional_edges("planner", route_after_planner,
                             {"comparison_agent": "comparison_agent", "data_agent": "data_agent"})
graph.add_conditional_edges("data_agent", route_after_metrics,
                             {"rag_agent": "rag_agent", "summary_agent": "summary_agent"})
graph.add_conditional_edges("comparison_agent", route_after_metrics,
                             {"rag_agent": "rag_agent", "summary_agent": "summary_agent"})
graph.add_edge("rag_agent", "summary_agent")
graph.add_edge("summary_agent", END)

orchestrator = graph.compile()
print("Orchestrator graph compiled.")


Orchestrator graph compiled.


## 7. Evaluation Checkpoint — 3 Sample Questions

For each question: the retrieved FAQ chunks (if RAG ran) and the final
narrative answer.


In [13]:
def ask(question: str):
    print("Q:", question)
    result = orchestrator.invoke({"question": question})
    if result.get("rag_result"):
        print("\nRetrieved chunks:")
        for chunk, score in zip(result["rag_result"].chunks, result["rag_result"].scores):
            print(f"  (score={score}) {chunk.replace(chr(10), ' ')[:90]}...")
    print("\nFinal answer:\n", result["final_answer"])
    print("\n" + "=" * 80 + "\n")
    return result


_ = ask("What is our current CSAT and does it meet target?")


Q: What is our current CSAT and does it meet target?

Retrieved chunks:
  (score=1.188) Q: What is your CSAT target? A: We aim for a CSAT of 4.5+. Scores below 4.0 trigger a root...
  (score=1.782) Q: What is your average wait time? A: We target under 10 minutes for counter orders during...
  (score=1.792) Q: Do you offer delivery? A: Yes, delivery is available through our app and major third-pa...

Final answer:
 Across 10227 responses, CSAT is 63.8% with an average rating of 3.72/5. The most frequently mentioned themes were price, wait_time, food. For context, company policy notes: "Q: What is your CSAT target? A: We aim for a CSAT of 4.5+. Scores below 4.0 trigger a root-cause review with the operations team. Reviews below 3.5 for two consecutive weeks trigger a full site audit."




In [14]:
_ = ask("What are the top complaints this month and how do they compare to last month?")


Q: What are the top complaints this month and how do they compare to last month?

Retrieved chunks:
  (score=0.966) Q: How do you handle complaints? A: All complaints are escalated to the shift manager with...
  (score=1.629) Q: How is staffing adjusted for busy periods? A: Staffing schedules are built two weeks in...
  (score=2.0) Q: What is your CSAT target? A: We aim for a CSAT of 4.5+. Scores below 4.0 trigger a root...

Final answer:
 In the current period, CSAT was 63.8% (avg rating 3.72) across 10227 responses, versus 65.6% (avg rating 3.75) in the previous period -- CSAT declined by 1.8 points month-over-month. The most common themes this period were price, wait_time, food. For context, company policy notes: "Q: How do you handle complaints? A: All complaints are escalated to the shift manager within 15 minutes. Refunds or replacements are offered for quality issues. Repeated complaints from the same customer within 30 days trigger a personal follow-up call from the regional ma

In [15]:
_ = ask("How does customer wait time feedback look, and what's our policy on it?")


Q: How does customer wait time feedback look, and what's our policy on it?



Retrieved chunks:
  (score=1.635) GreenLeaf Bistro -- Customer Experience FAQ  Q: What are your most popular menu items? A: ...
  (score=1.681) Q: How do you use customer survey data? A: Survey responses are reviewed weekly by the ope...
  (score=1.692) Q: What channels can customers use to leave feedback? A: Customers can respond via the mob...

Final answer:
 Across 10227 responses, CSAT is 63.8% with an average rating of 3.72/5. The most frequently mentioned themes were price, wait_time, food. For context, company policy notes: "GreenLeaf Bistro -- Customer Experience FAQ  Q: What are your most popular menu items? A: Our top sellers are the Avocado Toast, Garden Bowl, and Cold Brew Coffee. Seasonal items rotate every quarter based on customer feedback and local produce availability."




**Where retrieval worked well:** Question 1 (CSAT target) retrieved the
correct chunk with a strong margin — "CSAT" is a distinctive, low-frequency
term in the FAQ, so TF-IDF alone was enough to match it correctly.

**Where retrieval fell short:** Question 3 (wait time) retrieved the wrong
chunks. The query "customer wait time feedback... policy" shares common
words (*customer*, *feedback*) with several unrelated FAQ entries, and
those words happen to repeat more often in other chunks (e.g. the loyalty
and feedback-channels entries) than the word "wait" repeats in the actual
wait-time answer — so bag-of-words term frequency outweighs the one truly
relevant keyword. A real embedding model (sentence-transformers/OpenAI)
would very likely fix this, since it captures that "wait time" and "how
long we waited" are semantically close even without exact word overlap.
This is the clearest concrete case for upgrading past TF-IDF before any
real deployment.


## 8. Part 3 — Fine-Tuning Design (for the README)

*(This section answers the required 300–500 word design question. It's
included here as well as in `README.md` since it's easiest to review
alongside the code it refers to.)*

**Scenario:** omniSense needs to classify 10,000 free-text survey
responses/day into 8 sentiment+topic categories. GPT-4o is accurate but
too costly to run at that volume every day.

---

**1. Data strategy.** I'd start from the very metrics this notebook
already computes: use the current LLM/GPT-4o pipeline to *label* a sample
of real (or realistic synthetic) free-text responses across the 8
categories, stratified to avoid collapsing into the two or three most
common classes — survey text skews heavily positive, so "Negative"
categories need deliberate oversampling in the training set even though
they're rarer in production. I'd target roughly **1,500–3,000 labeled
examples** (150–350 per class) as a starting point for LoRA fine-tuning
on a strong open base model; classification into a fixed small label set
needs far fewer examples than open-ended generation. I'd hold out ~15% as
a fixed test set that is *never* touched during iteration, plus a rolling
weekly sample of new production data for drift checks.

**2. Model & technique.** A compact open model in the 3B–8B range (e.g.
Llama 3.1 8B or Qwen2.5 7B) is a strong base for this — the task is
classification, not open-ended reasoning, so a large model is overkill.
I'd use **QLoRA**: full fine-tuning is unnecessary and expensive for a
single classification head, and QLoRA's 4-bit quantization keeps training
on a single GPU feasible, which matters given the frontier-model cost
problem this project is trying to solve in the first place.

**3. Training pipeline.** Hugging Face `Trainer` with `peft` for the LoRA
adapters is the simplest path given the scope — no need for Axolotl or
LLaMA-Factory's extra features at this scale. I'd frame it as a
classification head over the 8 labels (or constrained generation of the
label string), train for a few epochs with early stopping on a held-out
validation split, and log everything to a simple experiment tracker (e.g.
Weights & Biases) so runs are comparable.

**4. Evaluation.** Per-class precision/recall/F1 (not just overall
accuracy — accuracy hides underperformance on rare classes like "Negative
- Wait Time"), plus a confusion matrix. I'd promote the fine-tuned model to
production only once it's within an agreed tolerance (e.g. within 2-3
points of macro-F1) of GPT-4o on the held-out test set *and* on the most
recent week of real traffic, not just the original test set — the two can
diverge as customer language shifts.

**5. Serving.** Serve the LoRA adapter alongside the existing LLM service
using a single base-model deployment with multiple adapters loaded (e.g.
via vLLM's multi-LoRA serving), routed by request type, so other
LLM-backed routes are untouched and the base model isn't duplicated per
adapter.

**6. Future-proofing.** Keep the labeling schema and prompt/label mapping
in a versioned config file, not hardcoded — so adding a 9th category or
adapting to a different client's category set doesn't require touching
training code, only the config and a fresh labeled batch.
